# Rol 5. Líder de lenguaje natural (Javier)

**Task 1.5.** En este notebook se analiza la auditoría de narrativas redactadas por un modelo de lenguaje. Se calculan la tasa de alucinación y el tiempo ahorrado, y se redacta la sección 5 del memo.

Se agrega la raíz del repositorio al `sys.path` para importar los módulos de `src/`. Los datos del enunciado están en `src/case_data.py`.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from src import case_data as cd

from src.metrics import hallucination_rate

## 1. Tasa de alucinación

La tasa de alucinación es la fracción de narrativas auditadas que contienen al menos un hecho inventado. La redacción manual no tiene conteo en el enunciado (`NaN`).

In [2]:
audit = cd.narrative_audit()
audit["tasa_alucinacion"] = [
    hallucination_rate(f, a) if pd.notna(f) else None
    for f, a in zip(audit["con_hechos_inventados"], audit["auditadas"])
]
audit

,configuracion,con_hechos_inventados,auditadas,minutos_por_narrativa,tasa_alucinacion
0,"Sin contexto, T=0.7",6.0,40.0,1.5,0.15
1,"Con contexto, T=0.2",2.0,40.0,1.5,0.05
2,Manual,NaN,NaN,6.0,NaN


### a. Tasas y mecanismo de la alucinación

La configuración sin contexto y con temperatura 0.7 alucina en el 15% de las narrativas. Con las transacciones en el contexto y temperatura 0.2 la tasa baja a 5%.

Un modelo de lenguaje genera el texto palabra por palabra (token por token). En cada paso elige el siguiente token a partir de una distribución de probabilidad aprendida, condicionada por el texto anterior. El modelo optimiza que el texto sea **plausible**, no que sea **verdadero**. Si las transacciones no están en el contexto, completa la narrativa con valores típicos de este tipo de textos, como una transacción de USD 9,800 que no existe.

- **El contexto** reduce la alucinación porque los montos y fechas reales quedan disponibles, y el token más probable pasa a ser el que copia esos datos.
- **La temperatura baja** concentra la probabilidad en los tokens más probables y reduce la elección de tokens poco probables, que es donde aparecen los datos inventados.

Ninguna de las dos medidas elimina el problema, porque persiste un 5%.

## 2. Tiempo ahorrado

In [3]:
manual = audit.loc[audit["configuracion"] == "Manual", "minutos_por_narrativa"].item()
assisted = audit.loc[audit["configuracion"] == "Con contexto, T=0.2", "minutos_por_narrativa"].item()
saved = manual - assisted
print(f"Tiempo ahorrado por narrativa: {saved} min ({saved / manual:.0%} del tiempo manual)")

Tiempo ahorrado por narrativa: 4.5 min (75% del tiempo manual)


### b. Tiempo ahorrado y condiciones de uso

Cada narrativa asistida, ya revisada, toma 1.5 minutos frente a 6 de la redacción manual. Se ahorran 4.5 minutos por narrativa, el 75% del tiempo.

El modelo de lenguaje se usaría en el proceso regulado solo bajo estas condiciones.

1. **Solo como borrador.** El oficial de cumplimiento revisa, corrige y firma cada narrativa. El modelo nunca envía nada al regulador.
2. **Configuración controlada.** Siempre con las transacciones de la alerta en el contexto y temperatura baja (0.2 o menor).
3. **Verificación automática.** Cada monto, fecha y cuenta citada en la narrativa se compara con la secuencia del remitente antes de llegar al oficial.
4. **Trazabilidad y auditoría.** Se registran la versión del modelo, la instrucción usada y las correcciones del oficial, y se repite la auditoría de 40 narrativas de forma periódica.

### c. Sección 5 del memo

> **5. Lenguaje natural.** Se recomienda usar el modelo de lenguaje solo como borrador, con las transacciones de la alerta en el contexto y temperatura 0.2. Esta configuración reduce la alucinación de 15% a 5% y ahorra 4.5 minutos por narrativa (75%). Como el 5% de las narrativas aún contiene hechos inventados, cada cifra citada se verifica de forma automática contra la secuencia del remitente. El oficial de cumplimiento revisa y firma cada narrativa y responde por su contenido.